# Preparing Data and Retrieval for a RAG System in Langchain

In [12]:
# Step 1: Install required packages
print("Installing packages (this may take 2-3 minutes)...")
!pip install -q langchain langchain-community sentence-transformers chromadb transformers torch accelerate bitsandbytes
print("\nInstallation complete!\n")
!o

Installing packages (this may take 2-3 minutes)...

Installation complete!



'o' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
!pip install chromadb

In [14]:
# Step 2: Import libraries
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

print("Starting Complete RAG Pipeline Demo")
print("="*60)


Starting Complete RAG Pipeline Demo


In [15]:
# Step 3: Create sample documents (your knowledge base)
print("\nStep 1: Creating knowledge base...")
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, and chains.",
             metadata={"source": "doc1"}),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retrieved based on similarity, then an LLM generates an answer using those documents as context.",
             metadata={"source": "doc2"}),
    Document(page_content="Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning.",
             metadata={"source": "doc3"}),
    Document(page_content="ChromaDB is an open-source embedding database that works well with LangChain. It's lightweight and perfect for development.",
             metadata={"source": "doc4"}),
    Document(page_content="HuggingFace provides thousands of pre-trained models that can run locally or in the cloud. Popular models include FLAN-T5, GPT-2, and Llama.",
             metadata={"source": "doc5"}),
    Document(page_content="Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.",
             metadata={"source": "doc6"}),
]


Step 1: Creating knowledge base...


In [16]:
# Step 4: Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)
print(f"Created {len(splits)} document chunks")

Created 6 document chunks


In [17]:
# Step 5: Create embeddings
print("\nStep 2: Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")


Step 2: Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6052.13it/s]


Embedding model loaded


In [18]:
%pip install --upgrade pip chromadb langchain-chroma

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="rag_demo"
)
print("Vector database created")

# Set up retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}  # Retrieve top 2 most relevant chunks
)

Note: you may need to restart the kernel to use updated packages.
Vector database created


In [ ]:
# Step 8: Load LLM (using a small model
print("\n Step 4: Loading Language Model...")
print("   (Using FLAN-T5 - large model)")

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)


 Step 4: Loading Language Model...
   (Using FLAN-T5 - large model)


## Connecting the Language  Model and Building the complete RAG pipeline 

In [24]:

# Create pipeline directly
pipe = pipeline(
    "text-generation",  # Updated task name
    model=model_name,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device=0 if torch.cuda.is_available() else -1,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Wrap in LangChain
llm = HuggingFacePipeline(pipeline=pipe)
print("Language model loaded")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 5149.24it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalL

Language model loaded


In [25]:
# Step 9: Create custom prompt template
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context: {context}

Question: {question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [26]:
# Step 10: Create RAG chain
print("\n Building RAG chain...")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)
print(" RAG chain ready!")


 Building RAG chain...
 RAG chain ready!


In [28]:
# Step 11: Test the complete RAG system
print("\n" + "="*60)
print(" COMPLETE RAG DEMONSTRATION")
print("="*60)

def ask_question(query):
    """
    Complete RAG pipeline: Retrieval + Augmented + Generation
    """
    print(f"\n Question: {query}")
    print("\n Step 1: Retrieving relevant documents...")

    result = qa_chain({"query": query})

    # Show retrieved documents
    print(f"Retrieved {len(result['source_documents'])} documents:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"\n   {i}. {doc.page_content[:150]}...")
        print(f"      Source: {doc.metadata['source']}")

    # Show generated answer
    print(f"\n Step 2: Generating answer using LLM...")
    print(f"\n Answer: {result['result']}")
    print("\n" + "-"*60)

# Run demo queries
print("\n Running Demo Queries...\n")

ask_question("What is RAG?")

ask_question("What are embeddings and why are they useful?")

ask_question("Which database should I use for vector storage?")



 COMPLETE RAG DEMONSTRATION

 Running Demo Queries...


 Question: What is RAG?

 Step 1: Retrieving relevant documents...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved 2 documents:

   1. RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retriev...
      Source: doc2

   2. RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retriev...
      Source: doc2

 Step 2: Generating answer using LLM...

 Answer: Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context: RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retrieved based on similarity, then an LLM generates an answer using those documents as context.

RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. F

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved 2 documents:

   1. Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings....
      Source: doc6

   2. Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings....
      Source: doc6

 Step 2: Generating answer using LLM...

 Answer: Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context: Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.

Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.

Question: What are embeddings and why are they useful?

Answer:

------------------------------------------------------------

 Question: Which database should I use for vector st

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved 2 documents:

   1. Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning....
      Source: doc3

   2. Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning....
      Source: doc3

 Step 2: Generating answer using LLM...

 Answer: Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context: Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning.

Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning.

Question: Which database should I use for vector storage?

Answer:

-------------------------------